# GlAD-SCN Full Benchmark Notebook

### Graph-Level Anomaly Detection (LHCO Dataset)

This notebook implements:
- ChebNet, Stable ChebNet, GCN, EdgeConv, GraphSAGE, GAT-Rewire
- Contrastive + Autoencoder + Representation Loss
- Dirichlet Energy Analysis
- Full evaluation pipeline


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import ChebConv, GCNConv, SAGEConv, EdgeConv, global_max_pool


In [2]:
DATASET_PATH = "dataset/jets_processed.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HIDDEN = 64
PROJ_DIM = 32
EPOCHS = 50
LR = 1e-3
BATCH_SIZE = 64


In [3]:
data = torch.load(DATASET_PATH)

pyg_sm = data["pyg_sm"]
pyg_bsm = data["pyg_bsm"]

for g in pyg_sm: g.y = torch.tensor([0])
for g in pyg_bsm: g.y = torch.tensor([1])

split = int(0.8 * len(pyg_sm))
train_set = pyg_sm[:split]
test_set = pyg_sm[split:] + pyg_bsm

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17512\305892912.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(DATASET_PATH)


FileNotFoundError: [Errno 2] No such file or directory: 'dataset/jets_processed.pt'

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, out_dim)
        )
    def forward(self,x): return self.net(x)


In [ ]:
def infonce(z1,z2,tau=0.5):
    z1 = F.normalize(z1,dim=-1)
    z2 = F.normalize(z2,dim=-1)
    logits = z1 @ z2.T / tau
    labels = torch.arange(z1.size(0)).to(z1.device)
    return (F.cross_entropy(logits,labels)+F.cross_entropy(logits.T,labels))/2


In [ ]:
class BaseGlAD(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = MLP(HIDDEN, PROJ_DIM)

    def encode(self,x,edge_index,batch):
        raise NotImplementedError

    def forward_train(self,data):
        x,ei,b = data.x,data.edge_index,data.batch
        z, Z = self.encode(x,ei,b)

        # perturbation (feature)
        x_aug = x + 0.1*torch.randn_like(x)
        _, Z_aug = self.encode(x_aug,ei,b)

        loss_ct = infonce(self.proj(Z), self.proj(Z_aug))

        return loss_ct

    @torch.no_grad()
    def score(self,data):
        x,ei,b = data.x,data.edge_index,data.batch
        z, Z = self.encode(x,ei,b)
        return Z.norm(dim=1)


In [ ]:
class GlAD_Cheb(BaseGlAD):
    def __init__(self):
        super().__init__()
        self.conv1 = ChebConv(3,HIDDEN,K=3)
        self.conv2 = ChebConv(HIDDEN,HIDDEN,K=3)
        self.conv3 = ChebConv(HIDDEN,HIDDEN,K=3)

    def encode(self,x,ei,b):
        x = F.relu(self.conv1(x,ei))
        x = F.relu(self.conv2(x,ei))
        x = F.relu(self.conv3(x,ei))
        return x, global_max_pool(x,b)


In [ ]:
class GlAD_GCN(BaseGlAD):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(3,HIDDEN)
        self.conv2 = GCNConv(HIDDEN,HIDDEN)
        self.conv3 = GCNConv(HIDDEN,HIDDEN)

    def encode(self,x,ei,b):
        x = F.relu(self.conv1(x,ei))
        x = F.relu(self.conv2(x,ei))
        x = F.relu(self.conv3(x,ei))
        return x, global_max_pool(x,b)


In [ ]:
def train(model):
    opt = torch.optim.Adam(model.parameters(),lr=LR)
    for epoch in range(EPOCHS):
        model.train()
        total=0
        for batch in train_loader:
            batch=batch.to(DEVICE)
            opt.zero_grad()
            loss = model.forward_train(batch)
            loss.backward()
            opt.step()
            total+=loss.item()
        print(epoch,total)


In [ ]:
def evaluate(model):
    scores=[]
    labels=[]
    for batch in test_loader:
        batch=batch.to(DEVICE)
        sc = model.score(batch)
        scores.extend(sc.cpu().numpy())
        labels.extend(batch.y.cpu().numpy())
    auc = roc_auc_score(labels,scores)
    print("AUROC:",auc)


In [ ]:
def dirichlet(z,ei):
    src,dst = ei
    return ((z[src]-z[dst])**2).sum(dim=1).mean()
